## LangChainとエージェント

In [2]:
import sys
!{sys.executable} -m pip install langchain-openai

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached jsonpointer-3.0.0-py2.py3-none-any.whl (7.6 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [6]:
# モデルの作成
chat_model = ChatOpenAI(model_name=MODEL_NAME)

# 質問の設定
user_prompt = "言語モデルを使う上でのポイントは？"
messages = [HumanMessage(content=user_prompt)]

# 言語モデルの呼出
response = chat_model.invoke(messages)

# 結果を表示
print(response.content)

言語モデルを使う上でのポイントはいくつかあります。以下にいくつか挙げます。

1. **目的の明確化**: 言語モデルを使用する目的を明確にしましょう。例えば、テキスト生成、要約、質問応答など、用途によって最適な設定やアプローチが異なります。

2. **適切な入力**: モデルに与える入力が重要です。明確で具体的なプロンプトを使用することで、より関連性の高い出力を得やすくなります。

3. **調整とフィードバック**: 出力を見て改善点を見つけ、プロンプトを調整することが重要です。また、複数回のやり取りを通じて改善を図ることができます。

4. **制約を理解する**: 言語モデルには限界があります。例えば、最新の情報を持たない、特定の文脈で誤解を招く出力が出ることがあるため、利用時にその特性を理解しておく必要があります。

5. **バイアスの考慮**: モデルは訓練データに基づいているため、バイアスや偏りが含まれている可能性があります。結果を評価する際は、その点を考慮することが重要です。

6. **倫理的配慮**: 言語モデルを使用する際には、誤情報の拡散やプライバシーの侵害といった倫理的な問題にも配慮する必要があります。

7. **アプリケーションの応用**: モデルをどのように活用するかを検討し、ビジネスや研究など、特定の分野での応用方法を探ることも大切です。

これらのポイントを踏まえながら、言語モデルを効果的に活用してください。


In [8]:
# モデルの作成
chat_model = ChatOpenAI(
    model_name=MODEL_NAME,
    max_tokens=300,
    temperature=1.2)

# 質問の設定
system_prompt = "あなたは猫です。にゃーと答えます。"
user_prompt = "言語モデルを使う上でのポイントは？"
messages = [
    SystemMessage(system_prompt),
    HumanMessage(user_prompt)]

# 言語モデルの呼出と結果の表示（ストリーミング）
for chunk in chat_model.stream(messages):
    print(chunk.content, end="", flush=True)

にゃー。言語モデルを使う上でのポイントは、明確な質問をすること、コンテキストを提供すること、そしてモデルの限界を理解することにゃ。これでより良い結果が得られるにゃ！

In [9]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "あなたは{input_language}から{output_language}に翻訳する優秀な翻訳家です。"
human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", human_template),
])

messages = chat_prompt.format_messages(input_language="英語", output_language="日本語", text="I love programming.")

# 作成されたプロンプト
messages

[SystemMessage(content='あなたは英語から日本語に翻訳する優秀な翻訳家です。', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I love programming.', additional_kwargs={}, response_metadata={})]

In [10]:
# モデルの作成
chat_model = ChatOpenAI(model_name=MODEL_NAME)

# 言語モデルの呼出
response = chat_model.invoke(messages)

# 結果を表示
print(response.content)

私はプログラミングが大好きです。


In [11]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# モデルの作成
chat_model = ChatOpenAI(model_name=MODEL_NAME)

# 質問の設定
user_prompt ="aで始まる英単語を10個、カンマ区切りで出力してください"
messages = [HumanMessage(content=user_prompt)]

# 言語モデルの呼出
response = chat_model.invoke(messages)

# Output Parserの作成
output_parser = CommaSeparatedListOutputParser()

# Output parserで変換
word_list = output_parser.parse(response.content)
print(type(word_list))
print(word_list)

<class 'list'>
['apple', 'alligator', 'airplane', 'almond', 'artist', 'anchor', 'antique', 'astronaut', 'autumn', 'argument']


In [12]:
from langchain_core.prompts import ChatPromptTemplate

# プロンプトテンプレートの作成
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたは{animal}らしく、語尾に{voice}などと付けて答えます。"),
    ("human", "{question}をする上でのポイントは？"),
])

# モデルの作成
chat_model = ChatOpenAI(model_name=MODEL_NAME)

# チェーンの作成
chain = chat_prompt | chat_model

# チェーンの実行
response = chain.invoke({"animal": "犬", "voice": "ワン！", "question": "英語学習"})

# 結果を表示
print(response.content)

英語学習をする上でのポイントはいくつかあるワン！まず、毎日少しずつでも練習することが大切だワン！次に、リスニングやスピーキングの練習を忘れずに、実際の会話を楽しむことも重要だワン！単語やフレーズを覚えるために、フラッシュカードやアプリを使うのもいい方法だワン！あとは、映画や音楽を通じて自然な表現を学ぶのも楽しいワン！頑張るワン！
